<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🛠️ 在 LangChain / LangGraph 中定义 Tool 的 3 种方式

**一句话定位**:Tool 本质上就是一个暴露了 `name`、`description`、输入 schema 的函数或类——LLM 不会读你的代码,只会读这三样东西来决定**什么时候调用**、**怎么传参**。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**为什么 docstring 和类型提示这么重要?**

LLM 看不到你的函数实现,它能看到的只有:
- `name`(工具叫什么)
- `description`(什么时候该用它——通常直接取自 docstring)
- 输入 schema(需要传哪些参数、什么类型)

**docstring 写得含糊,LLM 调用工具的时机和参数就会跟着含糊**。这是三种定义方式共同的底层逻辑,区别只在于“这三样东西怎么生成”。

</div>

### 1️⃣ `@tool` 装饰器 —— 大多数场景下的首选

给函数加一个 `@tool`(来自 `langchain_core.tools`),最快、最 Pythonic 的写法:

| 属性 | 从哪来 |
|---|---|
| `name` | 默认 = 函数名 |
| `description` | 直接取自函数的 docstring |
| 输入 schema | 自动从 Python 类型提示(type hints)推断 |

In [ ]:
from langchain_core.tools import tool

@tool
def calculate_multiply(a: int, b: int) -> int:
    """Multiply two integers together. Use this whenever you need to compute a product."""
    return a * b

In [ ]:
# 看看装饰器自动帮你生成了什么
print(calculate_multiply.name)         # "calculate_multiply"
print(calculate_multiply.description)  # "Multiply two integers together..."
print(calculate_multiply.args)         # {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}

### 2️⃣ `@tool` + Pydantic —— 参数复杂时用

当工具需要**嵌套参数、取值约束、或者每个字段都要配一段说明**时,给 `@tool` 搭配一个 Pydantic 模型,通过 `args_schema` 传进去。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:这一步的目的不是“让代码更规范”,而是**防止 LLM 传出结构不对/取值非法的参数**——每个 `Field(description=...)` 都是在给 LLM 的输入做前置校验说明书。

</div>

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# 1. 定义输入 schema
class WeatherInput(BaseModel):
    location: str = Field(description="The city and state, e.g. San Francisco, CA")
    unit: str = Field(default="celsius", description="The temperature unit, either 'celsius' or 'fahrenheit'")

# 2. 把 schema 传进装饰器
@tool(args_schema=WeatherInput)
def get_current_weather(location: str, unit: str) -> str:
    """Fetch the real-time weather information for a given location."""
    # 这里换成你真正的 API 调用逻辑
    return f"The weather in {location} is 22 degrees {unit}."

### 3️⃣ 继承 `BaseTool` —— 需要完全掌控时用

想要更细粒度地控制执行逻辑、维护内部状态、自定义 callback、或者单独实现异步追踪,就继承 `BaseTool` 写一个类。

代价是**代码量明显变多**——跟我们之前聊 CrewAI 的 `@tool` vs `BaseTool` 时的结论是一个道理:**简单工具优先用装饰器,需要管理状态/复杂行为才升级成类**。

In [ ]:
from typing import Type
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query: str = Field(description="The search query to look up")

class CustomSearchTool(BaseTool):
    name: str = "custom_search_tool"
    description: str = "Useful for when you need to answer questions about current events."
    args_schema: Type[BaseModel] = SearchInput

    def _run(self, query: str) -> str:
        """Use the tool synchronously."""
        return f"Search results for: {query}"

    async def _arun(self, query: str) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("Async not implemented yet")

# 实例化
search_tool = CustomSearchTool()

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **小提醒**:`BaseTool` 是 Pydantic `BaseModel` 的子类,所以 `name: str = "..."` 这种**带类型注解**的写法(而不是老教程里常见的 `name = "..."`)在 Pydantic v2 下才是正确、稳定的写法,务必保留类型注解。

</div>

### 📐 三种方式怎么选

| 方式 | 代码量 | 适用场景 |
|---|---|---|
| `@tool` | 最少 | 参数简单、无内部状态的大多数工具 —— **默认选它** |
| `@tool(args_schema=...)` | 中等 | 参数多/需要约束取值范围/需要防止 LLM 传错结构 |
| 继承 `BaseTool` | 最多 | 需要维护状态、自定义异步行为、接入复杂 callback 体系 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

### ⚠️ 设计工具的三条最佳实践

1. **docstring 要写得像说明书,而不是注释**:明确说清楚“什么情况下该调用这个工具”“它期望收到什么”——LLM 是把 docstring 当操作手册读的。
2. **命名用 `snake_case`**:比如 `web_search` 而不是带空格或特殊字符的名字——部分上游 LLM provider 对工具名里带空格会直接抛校验错误。
3. **错误处理要“确定性”**:把有风险的代码(API 调用、数学逻辑)包进 `try-except`,把错误信息作为**字符串返回**,而不是让异常直接炸穿。这样 LLM 能“看懂”这次调用失败了,下一轮会尝试修正参数重试,而不是让整个程序崩溃。

</div>

### 🔗 下一步:把工具绑定给 LLM

定义好工具之后,用 `.bind_tools()` 让 LLM 知道它可以用哪些工具:

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools([calculate_multiply])

### 🧩 细节:不加 `@tool` 的普通函数也能用

`bind_tools` 和 `ToolNode` 都**直接接受带类型提示 + docstring 的普通函数**,内部自动转成 Tool。

| 环节 | 对普通函数做了什么 |
|---|---|
| `llm.bind_tools([fn])` | 转成 JSON schema 发给模型:函数名 → `name`,docstring → `description`,类型提示 → 参数 schema(**只传说明书,不执行**) |
| `ToolNode([fn])` | 初始化时自动包装成 Tool(`create_tool`);运行时读 `AIMessage.tool_calls` → 调用函数 → 生成带 `tool_call_id` 的 `ToolMessage` |

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""      # 没有 @tool
    return a * b

llm_with_tools = llm.bind_tools([multiply])      # ✅ 可用
llm_with_tools.kwargs["tools"]                   # 查看实际发给模型的 schema
# ToolNode([multiply])                           # ✅ 同样可用

**什么时候才必须加 `@tool`?**

| 需求 | 普通函数 | 加 `@tool` |
|---|---|---|
| 只用 `bind_tools` + `ToolNode` | ✅ 够用 | 可选 |
| 手动 `multiply.invoke(...)` / 读 `.name` `.args` | ❌ 没这些属性 | ✅ |
| 自定义 `name` / `args_schema` / `return_direct` | ❌ | ✅ |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:`@tool` 不提供额外信息,只是**显式地提前做了这次转换**。普通函数若缺 docstring / 类型提示,模型拿到的 `description` 和参数 schema 就是空的——所以上面"docstring 要写得像说明书"对普通函数同样适用。

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**:无论哪种定义方式,`bind_tools()` 之后的使用方式是统一的——定义只是在“生产”`name`/`description`/schema` 这三样东西,真正决定工具用得好不好的,是你的 docstring 写得够不够清楚。`.bind_tools()` 只是把工具的“说明书”告诉 LLM,真正执行调用、处理结果,还需要 `ToolNode`(LangGraph)或 `AgentExecutor`(LangChain)这类执行层来接手。

</div>